# SHM: train and evaluate rainflow damage with a calibrated positive scalar

Run the cells in order, or select **Run All**. This notebook loads the raw Train dataset, builds features, trains new models, and prints the measured scores. All implementation is in this notebook. No saved model or result file is required.

**Local data:** copy `SHM` from the supplied `PS3/02_Datasets` bundle into this folder's `data/`, retaining the Train names below. Data stays local and is ignored by Git.

```text
SHM/
  train.ipynb
  data/
    Train_Labels.csv
    Train/
      train01.csv ... train64.csv
```

Use Python 3.11. Install the pinned CPU libraries once in the notebook's Python environment:
```python
%pip install numpy==1.26.4 pandas==2.2.1 scipy==1.12.0 scikit-learn==1.4.1.post1 openpyxl==3.1.5 rainflow==3.2.0 threadpoolctl==3.4.0
```

The displayed scores are cross-validation on labelled **Train** data. The recipe was selected in earlier experiments on this corpus, so these are exploratory validation results. Official Test is never read. There are 64 targets, with unknown physical source groups and stress units. File IDs are not chronological. Whole waveforms are preserved for rainflow counting.

## 1. Imports and data location

This notebook reproduces the retained CUDA calibration: install PyTorch **2.5.1 with CUDA 12.4** in this environment and use a compatible NVIDIA GPU. Exponents are chosen by three-fold validation inside each outer fitting fold; 1,000 fixed Adam updates fit one positive scalar. There is no outer-validation early stopping.

In [1]:
from pathlib import Path
import sys, time, hashlib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits
HERE = Path.cwd() if Path.cwd().name == 'SHM' else Path.cwd()/'SHM'
DATA = HERE/'data'
assert DATA.is_dir(), f'Place the SHM Train dataset in {DATA} first (see the cell above).'
SEED = 17
started = time.perf_counter()
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)
print('Data:', DATA.resolve())
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch, rainflow
from sklearn.model_selection import KFold
assert torch.cuda.is_available(), 'The retained scalar calibration requires CUDA.'
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
print('PyTorch:',torch.__version__,'| GPU:',torch.cuda.get_device_name(0))

Python: 3.11.4 | NumPy: 1.26.4 | pandas: 2.2.1 | sklearn: 1.4.1.post1
Data: C:\000NebulaX\nebulax_p3\SHM\data


PyTorch: 2.5.1+cu124 | GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## 2. Extract rainflow moments from the complete raw Train signals

In [2]:
EXPONENTS=[2,3,4,5,6,8]
labels=pd.read_csv(DATA/'Train_Labels.csv').set_index('filename')
names=sorted(labels.index.tolist())
y=labels.loc[names,'damage'].to_numpy(float)
assert np.isfinite(y).all() and np.all(y>0)
moments,hashes=[],[]
for i,name in enumerate(names,1):
    assert Path(name).name==name
    path=DATA/'Train'/name
    hashes.append(hashlib.sha256(path.read_bytes()).hexdigest())
    raw=pd.read_csv(path,header=None,dtype=np.float64)
    assert raw.shape==(581120,1) and np.isfinite(raw.to_numpy()).all()
    cycles=np.asarray(list(rainflow.extract_cycles(raw.iloc[:,0].to_numpy())),dtype=np.float64)
    amplitudes,counts=cycles[:,0]/2,cycles[:,2]
    assert np.isin(counts,[.5,1.]).all()
    moments.append([float(np.sum(counts*amplitudes**m)) for m in EXPONENTS])
    if i%8==0: print(f'Extracted rainflow moments: {i}/{len(names)}',flush=True)
X=np.asarray(moments);assert np.isfinite(X).all() and np.all(X>0)
print('Files:',len(names),'| Samples per file:',len(raw),'| Candidate exponents:',EXPONENTS)

Extracted rainflow moments: 8/64


Extracted rainflow moments: 16/64


Extracted rainflow moments: 24/64


Extracted rainflow moments: 32/64


Extracted rainflow moments: 40/64


Extracted rainflow moments: 48/64


Extracted rainflow moments: 56/64


Extracted rainflow moments: 64/64


Files: 64 | Samples per file: 581120 | Candidate exponents: [2, 3, 4, 5, 6, 8]


## 3. Define grouped folds and fitting-only exponent selection

In [3]:
def grouped_folds(indices,count):
    unique=list(dict.fromkeys(hashes[int(i)] for i in indices))
    for fit_groups,val_groups in KFold(n_splits=count,shuffle=True,random_state=SEED).split(unique):
        fit_set={unique[i] for i in fit_groups};val_set={unique[i] for i in val_groups}
        assert not fit_set&val_set
        yield (np.array([i for i in indices if hashes[int(i)] in fit_set]),
               np.array([i for i in indices if hashes[int(i)] in val_set]))

def weighted_median(values,weights):
    order=np.argsort(values,kind='stable')
    return float(values[order][np.searchsorted(np.cumsum(weights[order]),weights.sum()/2)])

def mape(truth,prediction): return float(np.mean(np.abs(truth-prediction)/truth))
def score(truth,prediction): return max(0.,1-mape(truth,prediction))

def choose_exponent(indices):
    trials=[]
    for j,exponent in enumerate(EXPONENTS):
        targets,predictions=[],[]
        for fit,val in grouped_folds(indices,3):
            base=X[:,j]
            scale=weighted_median(y[fit]/base[fit],base[fit]/y[fit])
            targets.extend(y[val]);predictions.extend(scale*base[val])
        trials.append((mape(np.asarray(targets),np.asarray(predictions)),exponent,j))
    return min(trials)[2]

## 4. Train the positive scalar on CUDA
Normalization and initialization use only the fitting files. All 1,000 updates optimize fitting MAPE; held-out labels are used only for scoring.

In [4]:
def calibrate(fit,column):
    torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    moment=X[:,column]
    median=float(np.median(moment[fit]))
    normalized=moment[fit]/median
    initial=float(np.median(np.log(y[fit]/normalized)))
    x=torch.tensor(normalized,dtype=torch.float64,device='cuda')
    target=torch.tensor(y[fit],dtype=torch.float64,device='cuda')
    parameter=torch.nn.Parameter(torch.tensor(initial,dtype=torch.float64,device='cuda'))
    optimizer=torch.optim.Adam([parameter],lr=.005)
    for epoch in range(1000):
        optimizer.zero_grad(set_to_none=True)
        loss=torch.mean(torch.abs(torch.exp(parameter)*x-target)/target)
        assert bool(torch.isfinite(loss))
        loss.backward();optimizer.step()
    return {'exponent':EXPONENTS[column],'column':column,'median':median,
            'normalized_scale':float(torch.exp(parameter).detach())}

def predict(model,indices):
    # Keep the original normalization-then-multiplication order.
    x=torch.tensor(X[indices,model['column']]/model['median'],dtype=torch.float64,device='cuda')
    return (model['normalized_scale']*x).cpu().numpy()

all_indices=np.arange(len(y))
oof=np.full(len(y),np.nan);baseline=np.full(len(y),np.nan);fold_rows=[]
for fold,(fit,val) in enumerate(grouped_folds(all_indices,4),1):
    column=choose_exponent(fit)
    model=calibrate(fit,column)
    fit_pred=predict(model,fit);oof[val]=predict(model,val)
    baseline[val]=weighted_median(y[fit],1/y[fit])
    row={'fold':fold,'exponent':model['exponent'],'train_fit':score(y[fit],fit_pred),'validation':score(y[val],oof[val])}
    fold_rows.append(row)
    print(f"Fold {fold}: exponent={model['exponent']}, train={row['train_fit']:.6f}, validation={row['validation']:.6f}",flush=True)
assert np.isfinite(oof).all() and np.all(oof>0)
display(pd.DataFrame(fold_rows))

Fold 1: exponent=5, train=0.975008, validation=0.972997


Fold 2: exponent=5, train=0.977507, validation=0.966077


Fold 3: exponent=5, train=0.972988, validation=0.979320


Fold 4: exponent=5, train=0.973403, validation=0.978281


,fold,exponent,train_fit,validation
0,1,5,0.975008,0.972997
1,2,5,0.977507,0.966077
2,3,5,0.972988,0.979320
3,4,5,0.973403,0.978281


## 5. Print scores and train the final all-Train model

In [5]:
validation_score=score(y,oof)
train_score=np.mean([r['train_fit'] for r in fold_rows])
print(f'Mean fitting-fold score: {train_score:.9f}')
print(f'Out-of-fold score (1 - MAPE): {validation_score:.9f}')
print(f'Out-of-fold MAPE: {100*mape(y,oof):.6f}%')
print(f'Worst relative error: {100*np.max(np.abs(y-oof)/y):.6f}%')
print(f'Weighted-constant baseline score: {score(y,baseline):.9f}')
display(pd.DataFrame({'file':names,'truth':y,'prediction':oof,'relative_error':np.abs(y-oof)/y}).head(10))
final_model=calibrate(all_indices,choose_exponent(all_indices))
print('Final trained scalar model:',final_model)
print(f'Elapsed: {time.perf_counter()-started:.1f} seconds')

Mean fitting-fold score: 0.974726515
Out-of-fold score (1 - MAPE): 0.974168850
Out-of-fold MAPE: 2.583115%
Worst relative error: 13.697180%
Weighted-constant baseline score: 0.437322366


,file,truth,prediction,relative_error
0,train01.csv,0.103663,0.103581,0.000787
1,train02.csv,0.149209,0.147198,0.013475
2,train03.csv,0.111298,0.112774,0.013268
3,train04.csv,0.894736,0.911711,0.018972
4,train05.csv,0.054740,0.057174,0.044467
5,train06.csv,0.467359,0.470061,0.005781
6,train07.csv,0.028737,0.028534,0.007071
7,train08.csv,0.078418,0.079198,0.009941
8,train09.csv,0.033629,0.034025,0.011779
9,train10.csv,0.716510,0.711288,0.007288


Final trained scalar model: {'exponent': 5, 'column': 3, 'median': 71745509.56485668, 'normalized_scale': 0.09757910604104837}
Elapsed: 69.7 seconds
